# LABORATORIO CALIFICADO N° 03
## Fundamentos de Gestión de Datos — Semana 7
**Docente:** Pilar Rocío Sayán Mejía | **Semestre:** 2026-I

---
## CASO: Predicción de Morosidad en FinanZa
**Protagonista:** Sofía Quispe, Data Scientist
**Empresa:** FinanZa — Fintech de préstamos personales con 15,000 clientes activos

**Tareas de Sofía:**
1. Limpiar el dataset crediticio con pipeline profesional
2. Identificar variables que correlacionan con el monto aprobado
3. Construir modelo de regresión lineal con R² ≥ 0.65

---

## PASO 1: Generación del Dataset Crediticio de FinanZa

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

np.random.seed(42)
n = 600

ingreso_base = np.random.normal(3500, 1500, n).clip(800, 8000)
ingreso_base[np.random.choice(n, 25, replace=False)] = None  # nulos
# outliers extremos en 8 registros
outlier_idx = np.random.choice(n, 8, replace=False)
ingreso_arr = ingreso_base.copy()
for idx in outlier_idx:
    if ingreso_arr[idx] is not None:
        ingreso_arr[idx] = np.random.uniform(18000, 25000)

edad_arr = np.random.randint(22, 65, n).astype(float)
edad_arr[np.random.choice(n, 30, replace=False)] = np.nan

historial = np.random.randint(20, 100, n)
num_creditos = np.random.randint(0, 6, n)
monto_sol = np.random.uniform(1000, 20000, n).round(2)

# Variable objetivo: monto aprobado (dependiente de ingreso e historial)
ingreso_limpio = np.where(pd.isnull(ingreso_arr), 3500, ingreso_arr)
monto_aprobado = (0.55 * ingreso_limpio + historial * 45 + num_creditos * 200
                  + np.random.normal(0, 800, n)).clip(500, 18000).round(2)

df = pd.DataFrame({
    'id_cliente':        [f'FZ{i:04d}' for i in range(1, n+1)],
    'edad':              edad_arr,
    'ingreso_mensual':   ingreso_arr,
    'historial_pagos':   historial,
    'num_creditos_prev': num_creditos,
    'monto_solicitado':  monto_sol,
    'monto_aprobado':    monto_aprobado,
    'estado_pago':       np.random.choice(['Al_dia','Mora_leve','Mora_grave'],
                                          n, p=[0.70, 0.20, 0.10]),
})

print('Dataset FinanZa generado:')
print(f'  Shape: {df.shape}')
print(f'  Nulos por columna:')
print(df.isnull().sum())
display(df.head())

## PASO 2: Análisis Exploratorio de Datos (EDA)

In [ ]:
print('=== INFO ===')
df.info()
print('\n=== ESTADISTICAS ===')
display(df.describe())

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('FinanZa — Distribuciones del Dataset Crediticio', fontsize=13, fontweight='bold')

df['edad'].dropna().plot(kind='hist', bins=20, ax=axes[0,0], color='#2E75B6', edgecolor='white')
axes[0,0].set_title('Edad'); axes[0,0].set_xlabel('Años')

df['ingreso_mensual'].dropna().plot(kind='hist', bins=25, ax=axes[0,1], color='#70AD47', edgecolor='white')
axes[0,1].set_title('Ingreso Mensual (S/.)'); axes[0,1].set_xlabel('S/.')

df['monto_solicitado'].plot(kind='hist', bins=20, ax=axes[1,0], color='#ED7D31', edgecolor='white')
axes[1,0].set_title('Monto Solicitado (S/.)'); axes[1,0].set_xlabel('S/.')

df['monto_aprobado'].plot(kind='hist', bins=20, ax=axes[1,1], color='#4472C4', edgecolor='white')
axes[1,1].set_title('Monto Aprobado (S/.) — Variable Objetivo'); axes[1,1].set_xlabel('S/.')

plt.tight_layout(); plt.savefig('eda_distribuciones.png', dpi=150, bbox_inches='tight'); plt.show()

# Heatmap de correlaciones
plt.figure(figsize=(8,6))
num_cols = ['edad','ingreso_mensual','historial_pagos','num_creditos_prev','monto_aprobado']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Matriz de Correlación — FinanZa')
plt.tight_layout(); plt.savefig('correlacion.png', dpi=150, bbox_inches='tight'); plt.show()
print('Variable mas correlacionada con monto_aprobado:')
print(corr['monto_aprobado'].drop('monto_aprobado').abs().sort_values(ascending=False))

## PASO 3: Detección y Tratamiento de Outliers

In [ ]:
def detectar_outliers_iqr(serie):
    Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf, lim_sup = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    return lim_inf, lim_sup, ((serie < lim_inf) | (serie > lim_sup)).sum()

serie = df['ingreso_mensual'].dropna()
li, ls, n_out = detectar_outliers_iqr(serie)
print(f'Outliers en ingreso_mensual:')
print(f'  Limite inferior IQR: S/. {li:.0f}')
print(f'  Limite superior IQR: S/. {ls:.0f}')
print(f'  Registros fuera del rango: {n_out}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['ingreso_mensual'].plot(kind='box', ax=axes[0], color='#C00000')
axes[0].set_title('Ingreso Mensual — ANTES (con outliers)')

df_clean = df.copy()
df_clean['ingreso_mensual'] = df_clean['ingreso_mensual'].clip(lower=li, upper=ls)
df_clean['ingreso_mensual'].plot(kind='box', ax=axes[1], color='#1A7F37')
axes[1].set_title('Ingreso Mensual — DESPUES (outliers corregidos)')
plt.tight_layout(); plt.show()

# Z-score para monto_aprobado
z_scores = (df['monto_aprobado'] - df['monto_aprobado'].mean()) / df['monto_aprobado'].std()
print(f'\nRegistros con |z-score| > 3 en monto_aprobado: {(z_scores.abs() > 3).sum()}')

## PASO 4: Imputación y Normalización

In [ ]:
# Imputar nulos
mediana_edad = df_clean['edad'].median()
mediana_ingreso = df_clean['ingreso_mensual'].median()
df_clean['edad'] = df_clean['edad'].fillna(mediana_edad)
df_clean['ingreso_mensual'] = df_clean['ingreso_mensual'].fillna(mediana_ingreso)
print(f'Nulos restantes: {df_clean.isnull().sum().sum()}')

# Normalización Min-Max
cols_norm = ['edad','ingreso_mensual','historial_pagos','num_creditos_prev','monto_aprobado']
scaler = MinMaxScaler()
df_norm = df_clean.copy()
df_norm[cols_norm] = scaler.fit_transform(df_clean[cols_norm])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_clean['ingreso_mensual'].plot(kind='hist', bins=20, ax=axes[0], color='#ED7D31', edgecolor='white')
axes[0].set_title('Ingreso Mensual — ANTES de normalizar')
df_norm['ingreso_mensual'].plot(kind='hist', bins=20, ax=axes[1], color='#70AD47', edgecolor='white')
axes[1].set_title('Ingreso Mensual — DESPUES (Min-Max 0-1)')
plt.tight_layout(); plt.show()
print('Rango normalizado:')
print(df_norm[cols_norm].describe().loc[['min','max']])

## PASO 5: Entrenamiento del Modelo — Regresión Lineal

In [ ]:
X = df_clean[['ingreso_mensual','historial_pagos','num_creditos_prev','edad']]
y = df_clean['monto_aprobado']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

modelo = LinearRegression()
modelo.fit(X_train, y_train)

print('\nCOEFICIENTES DEL MODELO:')
for nombre, coef in zip(X.columns, modelo.coef_):
    print(f'  {nombre:25s}: {coef:.4f}')
print(f'  Intercepto: {modelo.intercept_:.4f}')

ecuacion = 'monto_aprobado = '
partes = [f'{coef:.2f}*{nombre}' for nombre, coef in zip(X.columns, modelo.coef_)]
print(f'\nEcuación del modelo: {ecuacion} + '.join(partes) + f' + {modelo.intercept_:.2f}')

## PASO 6: Evaluación del Modelo

In [ ]:
y_pred_train = modelo.predict(X_train)
y_pred_test  = modelo.predict(X_test)

r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = mse_test ** 0.5

print('METRICAS DE EVALUACION:')
print(f'  R² Train : {r2_train:.4f}')
print(f'  R² Test  : {r2_test:.4f}  <- META: >= 0.65')
print(f'  MSE Test : {mse_test:.2f}')
print(f'  RMSE Test: S/. {rmse_test:.2f}')
print(f'\nMeta cumplida: {"SI" if r2_test >= 0.65 else "NO"}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, y_pred_test, alpha=0.5, color='#2E75B6')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn,mx],[mn,mx],'r--',lw=2,label='Prediccion perfecta')
axes[0].set_xlabel('Valores Reales (S/.)'); axes[0].set_ylabel('Valores Predichos (S/.)')
axes[0].set_title(f'Reales vs Predichos (R²={r2_test:.3f})')
axes[0].legend()

residuos = y_test - y_pred_test
axes[1].hist(residuos, bins=25, color='#70AD47', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Distribución de Residuos')
axes[1].set_xlabel('Residuo (S/.)')
plt.tight_layout(); plt.savefig('modelo_evaluacion.png', dpi=150, bbox_inches='tight'); plt.show()

# Guardar en SQLite
conn = sqlite3.connect(':memory:')
df_clean.to_sql('creditos_limpio', conn, if_exists='replace', index=False)
pred_df = pd.DataFrame({'id_cliente': df_clean.iloc[X_test.index]['id_cliente'].values,
                         'real': y_test.values, 'predicho': y_pred_test})
pred_df.to_sql('predicciones', conn, if_exists='replace', index=False)
print('\nPredicciones guardadas en SQLite (tabla: predicciones)')
display(pred_df.head())

## ACTIVIDAD 3: Análisis Reflexivo

**A.** ¿Por qué es importante detectar y tratar outliers antes de entrenar el modelo?

*(Su respuesta aqui)*

---

**B.** ¿El modelo de Sofía cumplió R² ≥ 0.65? ¿Qué variable tuvo mayor coeficiente?

*(Su respuesta aqui)*

---

**C.** ¿Qué limitación tiene la regresión lineal para predecir morosidad? ¿Qué modelo alternativo usaría?

*(Su respuesta aqui)*

---

**D.** ¿Cómo ayuda el pipeline CRISP-DM a estructurar el trabajo de Sofía?

*(Su respuesta aqui)*

## CONCLUSIONES

1. *(Conclusión 1)*

2. *(Conclusión 2)*

3. *(Conclusión 3)*

---
**Docente:** Pilar Rocío Sayán Mejía | **FGD 2026-I** | **LAB-C3**